In [28]:
import json
import pandas as pd
import numpy as np
from datetime import datetime
import prettytable as pt
import os
import warnings
warnings.filterwarnings("ignore")



###################################################################################################
                                                                                                  #
from rhoova.Client import *                                                                       #   
#Register and get api key from https://app.rhoova.com/ for ClientConfig("api key", "api secret")  #
config = ClientConfig("", "")                                                                     #
api = Api(config)                                                                                 # 
                                                                                                  #
###################################################################################################


directory = os.path.normpath(os.getcwd() + os.sep + os.pardir)
yielddatadirectory=directory+"/data/yielddata/bonddefiniton.csv"
marketdatadirectory=directory+"/data/marketdata/marketdata.csv"


bonddefiniton = pd.read_csv(yielddatadirectory,sep=';')
marketdata = pd.read_csv(marketdatadirectory,sep=';')

data=bonddefiniton.merge(marketdata[['valuationDate','isincode','value']], left_on='isincode', right_on='isincode')
data["valuationDate"]="2021-03-05"
data['issueDate'] = data['issueDate'].apply(lambda x: datetime.strptime(x, "%d.%m.%Y").strftime("%Y-%m-%d"))
data['maturityDate'] = data['maturityDate'].apply(lambda x: datetime.strptime(x, "%d.%m.%Y").strftime("%Y-%m-%d"))


In [29]:
# Rhoova expects date,convention,calendar input  as a string. Frequency values are Semiannual, Annual etc.
#6M,1Y cant be accepted.All inputs for fixedratebond are in https://app.rhoova.com/docs#post-/tasks/fixed-rate-bond

#Create empty dictionary to fill bond definition. Dictionary name is not case sensitive.
#It has to be define fixedRateBondDefinition

fixedRateBondDefinition={}
fixedRateBondDefinition["issueDate"]="2014-02-19"
fixedRateBondDefinition["maturityDate"]="2045-02-17"
fixedRateBondDefinition["frequency"]="Semiannual"
fixedRateBondDefinition["coupon"]=0.06625
fixedRateBondDefinition["calendar"]="Turkey"
fixedRateBondDefinition["currency"]="USD"
fixedRateBondDefinition["dateGeneration"]="Backward"
fixedRateBondDefinition["dayCounter"]="Thirty360E"
fixedRateBondDefinition["businessDayConvention"]="Unadjusted"
fixedRateBondDefinition["maturityDayConvention"]="Unadjusted"
fixedRateBondDefinition["redemption"]=100
fixedRateBondDefinition["endOfMonth"]=True
fixedRateBondDefinition

{'issueDate': '2014-02-19',
 'maturityDate': '2045-02-17',
 'frequency': 'Semiannual',
 'coupon': 0.06625,
 'calendar': 'Turkey',
 'currency': 'USD',
 'dateGeneration': 'Backward',
 'dayCounter': 'Thirty360E',
 'businessDayConvention': 'Unadjusted',
 'maturityDayConvention': 'Unadjusted',
 'redemption': 100,
 'endOfMonth': True}

In [30]:
instruments={}
bonds={}
bonds["method"]="NelsonSiegel"
bonds["numberOfIterations"]=10000
bonds["accuracy"]=1e-6
bonds["simplexLambda"]=10
bonds["settlementDays"]=2
instruments["BONDS"]=bonds
instruments

{'BONDS': {'method': 'NelsonSiegel',
  'numberOfIterations': 10000,
  'accuracy': 1e-06,
  'simplexLambda': 10,
  'settlementDays': 2}}

In [31]:
#Define discount curve parameter for pricing bonds.
discountCurve={}
discountCurve["settlementDays"]=2
discountCurve["calendar"]="NullCalendar" 
discountCurve["currency"]="USD" # Currency is using filter yield data. Only same currency datas are filtered for bond calculation
discountCurve["dayCounter"]="Thirty360E" # Default value is Actual360
discountCurve["period"]="6M" # Can not use if instruments selected BONDS
discountCurve["intpMethod"]="Linear" # if method is not null, it will not used
discountCurve["instruments"]=instruments
discountCurve

{'settlementDays': 2,
 'calendar': 'NullCalendar',
 'currency': 'USD',
 'dayCounter': 'Thirty360E',
 'period': '6M',
 'intpMethod': 'Linear',
 'instruments': {'BONDS': {'method': 'NelsonSiegel',
   'numberOfIterations': 10000,
   'accuracy': 1e-06,
   'simplexLambda': 10,
   'settlementDays': 2}}}

In [32]:
# Fill bond trade information
positiondata={}
positiondata["notional"]=1000000
positiondata["valuationDate"]="2021-03-05"
positiondata["settlementDate"]="2021-03-09"
positiondata["buySell"]="Sell"
positiondata

{'notional': 1000000,
 'valuationDate': '2021-03-05',
 'settlementDate': '2021-03-09',
 'buySell': 'Sell'}

In [33]:
#Join all data
positiondata["fixedRateBondDefinition"]=fixedRateBondDefinition
positiondata["discountCurve"]=discountCurve
positiondata["yieldData"]=data.to_dict('r')


In [34]:
try:
    bondresult = api.createTask(CalculationType.FIXED_RATE_BOND, positiondata,True)
    result=json.loads(bondresult["result"])
except RhoovaError as e:
    e.printPretty()    

In [35]:
npvTable = pt.PrettyTable(['Parameters', 'Value'])
npvTable.add_row(['PV', result.get('pv')])
npvTable.add_row(['Clean Price', result.get('cleanPrice')])
npvTable.add_row(['Dirty Price', result.get('dirtyPrice')])
npvTable.add_row(['Accrued Amount', result.get('accruedAmount')])
npvTable.add_row(['Yield to Maturity', 100*result.get('yieldToMaturity')])
npvTable.add_row(['Duration', result.get('duration')])
npvTable.add_row(['Modified Duration', result.get('modifiedDuration')])
npvTable.add_row(['Macualay Duration', result.get('macaulayDuration')])
npvTable.add_row(['Convexity', result.get('convexity')])
npvTable.add_row(['Bps', result.get('bps')])
npvTable.align = 'r'
npvTable.float_format = '.3'
print(npvTable)



+-------------------+------------+
|        Parameters |      Value |
+-------------------+------------+
|                PV | 982725.862 |
|       Clean Price |     97.895 |
|       Dirty Price |     98.300 |
|    Accrued Amount |      0.405 |
| Yield to Maturity |      6.804 |
|          Duration |     12.150 |
| Modified Duration |     11.750 |
| Macualay Duration |     12.150 |
|         Convexity |    208.774 |
|               Bps |      0.118 |
+-------------------+------------+


In [36]:
cashflow=pd.DataFrame(result.get("data"))
cashflow[cashflow["termToMatByDay"]>0]

,fixingDate,accrualStart,accrualEnd,notional,currency,leg,payOrReceive,instrument,rate,zeroRate,spread,termToMatByDay,termToMatByYear,cashflow,discountFactor,cashflowPv
14,2021-02-15,2021-02-17,2021-08-17,1000000,USD,Fixed,Sell,Fixed Rate Bond,0.06625,0.027354,0,158,0.438889,33125.000000,0.987766,32719.756808
15,2021-08-13,2021-08-17,2022-02-17,1000000,USD,Fixed,Sell,Fixed Rate Bond,0.06625,0.030079,0,338,0.938889,33125.000000,0.971829,32191.850694
16,2022-02-15,2022-02-17,2022-08-17,1000000,USD,Fixed,Sell,Fixed Rate Bond,0.06625,0.032724,0,518,1.438889,33125.000000,0.953659,31589.952950
17,2022-08-15,2022-08-17,2023-02-17,1000000,USD,Fixed,Sell,Fixed Rate Bond,0.06625,0.035288,0,698,1.938889,33125.000000,0.933503,30922.284988
18,2023-02-15,2023-02-17,2023-08-17,1000000,USD,Fixed,Sell,Fixed Rate Bond,0.06625,0.037772,0,878,2.438889,33125.000000,0.911613,30197.164661
19,2023-08-15,2023-08-17,2024-02-19,1000000,USD,Fixed,Sell,Fixed Rate Bond,0.06625,0.040201,0,1060,2.944444,33493.055556,0.887971,29740.855194
20,2024-02-15,2024-02-19,2024-08-19,1000000,USD,Fixed,Sell,Fixed Rate Bond,0.06625,0.042523,0,1240,3.444444,33125.000000,0.863346,28598.321286
21,2024-08-15,2024-08-19,2025-02-17,1000000,USD,Fixed,Sell,Fixed Rate Bond,0.06625,0.044740,0,1418,3.938889,32756.944444,0.838013,27450.733223
22,2025-02-13,2025-02-17,2025-08-18,1000000,USD,Fixed,Sell,Fixed Rate Bond,0.06625,0.046913,0,1599,4.441667,33309.027778,0.811481,27029.646286
23,2025-08-14,2025-08-18,2026-02-17,1000000,USD,Fixed,Sell,Fixed Rate Bond,0.06625,0.048983,0,1778,4.938889,32940.972222,0.784692,25848.527637
